LV Trench Quantities by Width - Sendim
=
This notebook totals the length of the LV trenches of UFV Sendim by trench width, separately for the trenches that also carry DC cables (`cc` in the layer name) and the AC-only ones.

Unlike `Valas CA`, the key is the polyline **Global Width** (the trench width), not the circuit count read from the layer name.

**Workflow overview:**
1. Read the trench table exported from AutoCAD.
2. Split the trenches into DC (`cc` in the layer) and AC.
3. Sum the length per `Global Width` for each set.

**Input prerequisites (`valas ca.xls`, on the Desktop)** (the code assumes all of these, and it doesn't check them):
- One row per trench polyline, with the columns `Layer`, `Global Width` and `Length` (AutoCAD data extraction).
- Trenches with DC cables have lowercase `cc` in the layer name (e.g. `EMF_bt_vala_btca_01Xc_3cc_PST_16_Plot_13`). The match is case-sensitive, so `CC` isn't found.
- Every trench polyline has its width set as Global Width. Rows with an empty width are left out of the totals.

Reading the trenches and splitting DC / AC
=
- Reads the trench table and marks `CC = True` when the layer name contains `cc`.
- `df_valas_CC`: trenches that also carry DC cables. `df_valas_CA`: all the others.
- `math` and `numpy` are imported but not used.

In [1]:
import time

import pandas as pd

import math

import numpy as np

start = time.time()

df_valas = pd.read_excel(r"C:\Users\Usuario\Desktop\valas ca.xls")

df_valas['CC'] = df_valas['Layer'].str.contains('cc', na=False)

df_valas_CC = df_valas[df_valas['CC'] == True]
df_valas_CA = df_valas[df_valas['CC'] == False]

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero


Length per trench width
=
- `groupby('Global Width').sum()` gives the total `Length` per width, for the DC and the AC trenches.
- The `CC` column of the result is the number of trench polylines in each width (sum of `True`). The `Layer` column is just the concatenated layer names and can be ignored.
- Rows with an empty `Global Width` are dropped by `groupby` and aren't in the totals.
- The results are only printed, not saved.

In [2]:
df_cabos_CC = df_valas_CC.groupby('Global Width').sum()
df_cabos_CA = df_valas_CA.groupby('Global Width').sum()

print(df_cabos_CC)
print(df_cabos_CA)


end = time.time()
elapsed = end - start
print(elapsed)

                                                          Layer    Length  CC
Global Width                                                                 
0.5           EMF_bt_vala_btca_01Xc_3cc_PST_16_Plot_13EMF_bt...  104.2847   7
                                                          Layer     Length  CC
Global Width                                                                  
0.5           EMF_bt_vala_btca_01Xc_PST_08_Plot_07EMF_bt_val...  3906.4848   0
0.6                        EMF_bt_vala_btca_03Xc_PST_08_Plot_07    34.8450   0
0.8           EMF_bt_vala_btca_04Xc_PST_10_Plot_09EMF_bt_val...   456.7854   0
1.0           EMF_bt_vala_btca_04Xc_PST_08_Plot_08EMF_bt_val...   748.1419   0
1.2           EMF_bt_vala_btca_04Xc_PST_08_Plot_07EMF_bt_val...   754.7012   0
1.4           EMF_bt_vala_btca_05Xc_PST_08_Plot_08EMF_bt_val...   154.0037   0
1.5           EMF_bt_vala_btca_07Xc_PST_10_Plot_09EMF_bt_val...   177.6335   0
1.6                        EMF_bt_vala_btca_07Xc_PST_17